In [ ]:
# =========================
# nb_reports
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

from pyspark.sql import functions as F
from datetime import datetime
import json

output_base = f"/lakehouse/default/Files/output/period={period}/run_id={run_id}"

try:
    mssparkutils.fs.mkdirs(output_base)
except Exception:
    pass

# ---------- Load run data ----------
stage2 = spark.table("individual_dfm_consolidated").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)
stage3 = spark.table("aggregated_dfms_consolidated").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)
dq = spark.table("dq_results").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)
dq_rows = spark.table("dq_exception_rows").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if stage2.rdd.isEmpty():
    print("[nb_reports] No Stage 2 data for this run.")
    mssparkutils.notebook.exit("NO_DATA")

dfms = [r["dfm_id"] for r in stage2.select("dfm_id").distinct().collect()]

# ---------- Report 1 per DFM ----------
# Row-level exception counts by policyholder/check
for d in dfms:
    r1 = (
        dq_rows.filter(F.col("dfm_id") == d)
        .groupBy("dfm_id", "policyholder_number", "check_id", "failure_reason")
        .agg(F.count("*").alias("row_count"))
        .orderBy("policyholder_number", "check_id")
    )

    out = f"{output_base}/report1_{d}.csv"
    r1.coalesce(1).write.mode("overwrite").option("header", True).csv(out)

# ---------- Report 2 rollup ----------
r2 = (
    dq.groupBy("dfm_id", "check_id", "severity", "status")
    .agg(
        F.sum(F.coalesce(F.col("rows_failed"), F.lit(0))).alias("rows_failed"),
        F.sum(F.coalesce(F.col("total_rows_evaluated"), F.lit(0))).alias("rows_evaluated")
    )
    .orderBy("dfm_id", "check_id", "status")
)

r2.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_base}/report2_rollup.csv")

# ---------- reconciliation_summary.json ----------
stage3_totals = (
    stage3.groupBy("dfm_id")
    .agg(
        F.sum("bid_value_gbp").cast("double").alias("total_bid_value_gbp"),
        F.sum("cash_value_gbp").cast("double").alias("total_cash_value_gbp"),
        F.sum("accrued_interest_gbp").cast("double").alias("total_accrued_interest_gbp"),
        F.countDistinct("policyholder_number").alias("policy_count"),
        F.count("*").alias("aggregated_rows")
    )
)

stage2_rows = (
    stage2.groupBy("dfm_id")
    .agg(
        F.count("*").alias("stage2_rows"),
        F.sum(F.when(F.lower(F.col("include_flag")) == "include", F.lit(1)).otherwise(F.lit(0))).alias("stage2_include_rows")
    )
)

gate_state = (
    dq.groupBy("dfm_id")
    .agg(
        F.sum(
            F.when(
                (F.lower(F.col("severity")).isin("exception", "stop"))
                & (F.lower(F.col("status")) == "fail"),
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("blocking_fail_checks")
    )
)

summary_df = (
    stage2_rows
    .join(stage3_totals, "dfm_id", "left")
    .join(gate_state, "dfm_id", "left")
    .fillna(0)
)

summary_records = [r.asDict() for r in summary_df.collect()]

payload = {
    "period": period,
    "run_id": run_id,
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
    "dfm_summaries": summary_records,
}

json_path = f"{output_base}/reconciliation_summary.json"
mssparkutils.fs.put(json_path, json.dumps(payload, indent=2), True)

print(f"[nb_reports] Wrote outputs under: {output_base}")
mssparkutils.notebook.exit("OK")